# Session log — LLM-in-Sandbox on 2x T4

The **actual** exploratory session behind the results, kept because the dead ends
are the useful part: three of them changed the design.

For a clean re-run, use `colab_driver.ipynb` instead. This file is a record, not
a recipe.

Findings, in the order they were forced on us:

| # | probe | finding |
|---|---|---|
| 1 | `nvidia-smi` | 2x T4, **sm75** — fp16 only, no FA2/FP8/Marlin, `PHB` (no NVLink) |
| 2 | serve @ 32k | KV cache needs 4.50 GiB, 4.07 GiB free → context capped at 24576 |
| 3 | `transformers` | Colab ships 5.0; vLLM 0.11 calls the removed `all_special_tokens_extended` |
| 4 | PTY trials | bash spawns fine — so `shell died (exit -9)` was **our** bug, not the platform |
| 5 | raw PTY replay | sentinel *does* arrive → the bug was scanning the wrong buffer |
| 6 | `unshare` | **denied** on Colab (`Operation not permitted`) even as uid 0 |
| 7 | seccomp | **permitted** → enforced network ablation is recoverable |
| 8 | attention backend | FlexAttention 22 tok/s → TRITON_ATTN 51–69 tok/s per card |

## 1. What hardware did we actually get?

The answer reframes the whole project: `30 GB of VRAM` is **2x15 GB at sm75**, not one big card.

In [ ]:
import subprocess, sys, os, shutil
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version,compute_cap",
                      "--format=csv"],capture_output=True,text=True).stdout)
print("python", sys.version.split()[0], "| cpus", os.cpu_count())
import torch; print("torch", torch.__version__, "bf16", torch.cuda.is_bf16_supported())
for t in ["docker","bwrap","unshare","runsc"]:
    print(f"{t}: {shutil.which(t)}")
print("uid", os.getuid())

Result:

```
Tesla T4, 15360 MiB, compute_cap 7.5   (x2)
cpus 4 | torch 2.10.0+cu128 | uid 0
docker: None | bwrap: None | unshare: /usr/bin/unshare
```

`nvidia-smi topo -m` reported **PHB** between the cards — PCIe host bridge, no
NVLink. That is what later rules out tensor-parallel in favour of one replica
per GPU.

Note `is_bf16_supported()` returns **True** on a T4 and is misleading: there is
no native bf16 path on sm75.

## 2. Is there internet, and what's preinstalled?

In [ ]:
import socket, importlib.util, importlib.metadata as md
for host,port in [("pypi.org",443),("huggingface.co",443),("github.com",443)]:
    try:
        socket.create_connection((host,port), timeout=8); print(f"NET {host} OK")
    except Exception as e: print(f"NET {host} FAIL {e}")
for m in ["vllm","transformers","datasets","openai","flash_attn","xformers"]:
    print(f"PKG {m}", md.version(m) if importlib.util.find_spec(m) else "MISSING")

## 3. Install vLLM — detached

Launched with `nohup ... &` because the Colab MCP bridge has **no kernel interrupt**: a blocking cell wedges the kernel for everything else.

In [ ]:
import subprocess, os, textwrap
os.makedirs("/content/logs", exist_ok=True)
open("/content/install_vllm.sh","w").write(textwrap.dedent("""
pip install -q --upgrade pip
pip install "vllm==0.11.0" 2>&1 | tail -40
python -c "import torch, vllm; print('OK', vllm.__version__, torch.__version__)"
echo "___INSTALL_DONE___ rc=$?"
"""))
subprocess.Popen("nohup bash /content/install_vllm.sh > /content/logs/install_vllm.log 2>&1 &",
                 shell=True)
print("launched; poll the log")

In [ ]:
# Re-runnable poller.
import subprocess
print(subprocess.run("tail -c 1500 /content/logs/install_vllm.log", shell=True,
                     capture_output=True, text=True).stdout)

Installed vLLM 0.11.0, which **downgraded torch to 2.8.0+cu128** and pulled
in xformers 0.0.32.

## 4. Two serving failures, in order

**(a) `transformers` 5.0.** vLLM 0.11 calls `all_special_tokens_extended`, which
5.0 removed. vLLM's pin is `>=4.55.2`, satisfied by 5.0, so pip never downgraded
it:

```
AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended
```

**(b) KV cache at 32k.** With fp16 weights on a 15 GB card:

```
ValueError: To serve at least one request with max seq len (32768),
4.50 GiB KV cache is needed, larger than available (4.07 GiB).
```

Hence `transformers==4.56.2` and `--max-model-len 24576`.

## 5. The shell bug: `shell died (exit -9)`

All 16 shell tests failed. `-9` is SIGKILL. First question — is it the platform or us? These four trials spawn bash exactly as `PersistentShell` does.

In [ ]:
import os, pty, fcntl, termios, subprocess, time

def trial(name, use_setsid, use_ctty):
    m, s = pty.openpty()
    a = termios.tcgetattr(s); a[3] &= ~termios.ECHO; a[1] &= ~termios.ONLCR
    termios.tcsetattr(s, termios.TCSANOW, a)
    def pre():
        if use_setsid: os.setsid()
        if use_ctty: fcntl.ioctl(s, termios.TIOCSCTTY, 0)
    try:
        p = subprocess.Popen(["bash","--noprofile","--norc"], stdin=s, stdout=s, stderr=s,
                             preexec_fn=pre, close_fds=True, cwd="/tmp")
    except Exception as e:
        os.close(m); os.close(s); print(f"{name}: POPEN RAISED {e}"); return
    os.close(s); time.sleep(1.0)
    os.set_blocking(m, False)
    try: out = os.read(m, 4096)
    except Exception as e: out = repr(e).encode()
    print(f"{name}: poll={p.poll()} out={out[:80]!r}")
    if p.poll() is None: p.kill()
    os.close(m)

trial("A: setsid+ctty", True,  True)
trial("B: setsid only", True,  False)
trial("C: neither    ", False, False)
trial("D: ctty only  ", False, True)

```
A: setsid+ctty: poll=None out=b'\x1b[?2004hbash-5.1# '
B: setsid only: poll=None ...
C: neither    : poll=None  (+ "cannot set terminal process group")
D: ctty only  : POPEN RAISED  (setsid is a prerequisite)
```

**bash is alive in every viable variant.** So the SIGKILL was ours. Also visible:
`\x1b[?2004h` — readline's bracketed-paste escapes, which would land in the
model's observations. Fixed later with `--noediting`.

Next: replay the exact init sequence and watch the raw PTY bytes.

In [ ]:
import os, pty, fcntl, termios, subprocess, time, select

m, s = pty.openpty()
a = termios.tcgetattr(s); a[3] &= ~termios.ECHO; a[1] &= ~termios.ONLCR
termios.tcsetattr(s, termios.TCSANOW, a)
def pre(): os.setsid(); fcntl.ioctl(s, termios.TIOCSCTTY, 0)
p = subprocess.Popen(["bash","--noprofile","--norc"], stdin=s, stdout=s, stderr=s,
                     preexec_fn=pre, close_fds=True, cwd="/tmp")
os.close(s)

def drain(tag, wait=1.2):
    buf=b""; end=time.time()+wait
    while time.time()<end:
        if select.select([m],[],[],0.15)[0]:
            try: buf += os.read(m, 65536)
            except OSError: break
    print(f"[{tag}] alive={p.poll() is None} {buf[:200]!r}")

drain("after spawn")
os.write(m, b"set -m\nunset HISTFILE\nexport PS1= PS2=\n")
drain("after init")
os.makedirs("/tmp/.sandbox_lab", exist_ok=True)
open("/tmp/.sandbox_lab/cmd_1.sh","w").write(":")
os.write(m, b"source '/tmp/.sandbox_lab/cmd_1.sh'\n")
drain("after source")
os.write(m, b"printf '\\n__SENT_%d__\\n' \"$?\"\n")
drain("after printf")
p.kill(); os.close(m)

```
[after printf] alive=True b'\n__SENT_0__\n'
```

**The sentinel arrives.** So the framing was fine and the read loop was looking
in the wrong place. Two compounding bugs:

1. `_pump` scanned `_HeadTailBuffer._tail`, but that buffer fills **head-first**
   (32 KB). For any normal command the tail is empty, so the sentinel never
   matched and *every* command timed out.
2. On timeout, `_interrupt` escalated with `killpg(getpgid(bash_pid))` — but
   after `setsid()` bash **is** its own group leader, so that SIGKILLed the
   shell. Hence `-9`.

Fixes: an 8 KB sliding window scanned for the sentinel, and `os.tcgetpgrp()` to
read the *foreground* group from the terminal so the command is signalled and
bash is not.

## 6. Can we enforce ablations? `unshare` says no

In [ ]:
import subprocess
for cmd in ["unshare --fork --pid --mount true",
            "unshare --fork --pid --mount --net true",
            "unshare --net ip link set lo up"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    print(f"rc={r.returncode}  {cmd}   {r.stderr.strip()[:80]}")

```
rc=1  unshare --fork --pid --mount true   unshare failed: Operation not permitted
```

Denied even as **uid 0** — the container drops `CAP_SYS_ADMIN`. Namespace-based
ablation is off the table on Colab.

## 7. …but seccomp says yes

An unprivileged process may install a seccomp-BPF filter after setting `PR_SET_NO_NEW_PRIVS`. This is the probe that rescued the ablation design.

In [ ]:
probe = r"""
import ctypes, ctypes.util, os, socket, sys

class F(ctypes.Structure):
    _fields_ = [("code",ctypes.c_uint16),("jt",ctypes.c_uint8),
                ("jf",ctypes.c_uint8),("k",ctypes.c_uint32)]
class P(ctypes.Structure):
    _fields_ = [("len",ctypes.c_ushort),("filter",ctypes.POINTER(F))]

LD,JEQ,RET = 0x20,0x15,0x06
ALLOW, DENY = 0x7fff0000, 0x00050000|13
prog = (F*9)(F(LD,0,0,4), F(JEQ,0,5,0xC000003E), F(LD,0,0,0), F(JEQ,0,3,41),
             F(LD,0,0,16), F(JEQ,2,0,2), F(JEQ,1,0,10),
             F(RET,0,0,ALLOW), F(RET,0,0,DENY))
fprog = P(9, prog)
libc = ctypes.CDLL(ctypes.util.find_library("c"), use_errno=True)
assert libc.prctl(38,1,0,0,0) == 0                      # PR_SET_NO_NEW_PRIVS
assert libc.syscall(317,1,0,ctypes.byref(fprog)) == 0   # seccomp(SET_MODE_FILTER)
print("SECCOMP_INSTALLED")
try: socket.create_connection(("1.1.1.1",443),timeout=4); print("NET: REACHED (bad)")
except Exception as e: print("NET: blocked ->", type(e).__name__, e)
s = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM); s.close(); print("UNIX: OK")
open("/tmp/_sec.txt","w").write("x"); print("file write: OK")
"""
open("/tmp/seccomp_probe.py","w").write(probe)
import subprocess
print(subprocess.run(["python3","/tmp/seccomp_probe.py"], capture_output=True,
                     text=True, timeout=60).stdout)

```
SECCOMP_INSTALLED
NET: blocked -> PermissionError [Errno 13] Permission denied
UNIX: OK
file write: OK
```

Exactly the shape we want: **external** access gone, local IPC and files intact.
This became `sandbox_lab/sandbox/seccomp.py`.

## 8. Throughput: the attention backend is the whole game

The first sweep decoded at ~22 tok/s aggregate with GPU1 idle. vLLM had auto-selected **FlexAttention** (FA2 needs sm80+), which is a `torch.compile` fallback rather than a real kernel.

In [ ]:
import subprocess, textwrap

def launch(port, gpu, backend="TRITON_ATTN"):
    open(f"/content/serve_{port}.sh","w").write(textwrap.dedent(f"""
    export CUDA_VISIBLE_DEVICES={gpu}
    export VLLM_ATTENTION_BACKEND={backend}
    python -m vllm.entrypoints.openai.api_server \
      --model Qwen/Qwen3-4B-Instruct-2507 --served-model-name qwen3-4b \
      --dtype float16 --max-model-len 24576 --gpu-memory-utilization 0.93 \
      --enable-prefix-caching --enable-auto-tool-choice --tool-call-parser hermes \
      --max-num-seqs 16 --port {port} --host 127.0.0.1 2>&1
    """))
    subprocess.Popen(f"nohup bash /content/serve_{port}.sh > /content/logs/vllm_{port}.log 2>&1 &",
                     shell=True)

subprocess.run("pkill -f api_server; sleep 3", shell=True, capture_output=True)
launch(8000, 0); launch(8001, 1)
print("one replica per GPU, TRITON_ATTN")

In [ ]:
# Re-runnable: wait for servers, then measure real decode throughput.
import urllib.request, json, time
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

def up(port):
    try:
        json.load(urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=3))
        return True
    except Exception: return False

def bench(port, n_par=4, max_tok=256):
    c = OpenAI(base_url=f"http://127.0.0.1:{port}/v1", api_key="none")
    def one(i):
        r = c.chat.completions.create(model="qwen3-4b", max_tokens=max_tok, temperature=0.7,
            messages=[{"role":"user","content":f"Write a detailed explanation of topic {i}."}])
        return r.usage.completion_tokens
    t0 = time.time()
    with ThreadPoolExecutor(n_par) as ex: toks = sum(ex.map(one, range(n_par)))
    dt = time.time()-t0
    return f"{toks} tok in {dt:.1f}s = {toks/dt:.1f} tok/s"

for port in (8000, 8001):
    print(port, "UP" if up(port) else "not up", bench(port) if up(port) else "")

```
8000 UP 707 tok in 13.7s = 51.5 tok/s
8001 UP 797 tok in 11.6s = 68.9 tok/s
```

**~22 → ~120 tok/s total**: about 2.5–3x from the backend, 2x from using the
second card. One replica per GPU rather than TP=2, because `PHB` means no
NVLink and agent episodes parallelise across replicas for free.

## 9. End-to-end smoke test

Same question, both modes. This is where the loop's real failure modes showed up.

In [ ]:
import sys, json
sys.path.insert(0, "/content/llm-in-sandbox/src")
from openai import OpenAI
from sandbox_lab.agent import AgentConfig, SandboxAgent

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="none")
Q = "What is the remainder when 7^{1234} is divided by 1000? Give the final integer."

for mode in ["sandbox", "direct"]:
    cfg = AgentConfig(model="qwen3-4b", mode=mode, max_turns=12, max_tokens_per_turn=1024)
    t = SandboxAgent(client, cfg).run(Q, task_id="smoke",
                                      sandbox_root="/content/testbed_smoke", backend="local")
    print(f"\n== {mode} ==  answer={t.final_answer!r} stop={t.stop_reason} "
          f"turns={t.n_turns} gen_tokens={t.generated_tokens}")
    for tr in t.turns:
        for tc in tr.tool_calls:
            print(f"   turn{tr.index} -> {tc['name']}({json.dumps(tc['arguments'])[:100]})")
        if not tr.tool_calls:
            print(f"   turn{tr.index} content: {tr.content[-120:]!r}")

Both answered **849** (correct). Token counts: sandbox 812 vs direct 1660 —
a **0.49x** ratio, at the bottom of the paper's reported 0.49–0.84x band.

Two harness bugs surfaced here, both of which would have biased the sandbox arm
downwards:

1. It solved the task on **turn 0** with `pow(7,1234,1000)`, then burned every
   remaining turn emitting no tool call. Fixed: nudge once, stop on the second
   consecutive miss.
2. It wrote **`finish(849)` as prose** instead of emitting a tool call. Recovered
   by a deliberately narrow parser (only `finish`, only trailing), tested against
   inventing a call from a mid-sentence "I will call finish(x) once…".

The `direct` arm also returned `'FINAL ANSWER: 849'` — the label leaked into the
extracted answer and would have graded as wrong.

## 10. The sweep

Resumable: rerunning skips completed episodes, so a dropped runtime costs one episode, not the run.

In [ ]:
import subprocess, textwrap
open("/content/run_sweep.sh","w").write(textwrap.dedent("""
cd /content/llm-in-sandbox
export PYTHONPATH=/content/llm-in-sandbox/src HF_HUB_DISABLE_PROGRESS_BARS=1
python scripts/run_sweep.py \
  --benchmark mmlu_pro --n 25 --model qwen3-4b \
  --base-url http://127.0.0.1:8000/v1 http://127.0.0.1:8001/v1 \
  --modes direct sandbox --max-turns 20 --max-tokens-per-turn 2048 \
  --workers 8 --out /content/runs/pilot_qwen3-4b 2>&1
echo "___SWEEP_DONE___ rc=$?"
"""))
subprocess.Popen("nohup bash /content/run_sweep.sh > /content/logs/pilot.log 2>&1 &", shell=True)
print("sweep launched: 25/domain x 4 domains x 2 modes = 200 episodes")

In [ ]:
# Re-runnable progress poller.
import subprocess, json, collections, pathlib
print(subprocess.run("tail -c 1200 /content/logs/pilot.log", shell=True,
                     capture_output=True, text=True).stdout)
p = pathlib.Path("/content/runs/pilot_qwen3-4b/results.jsonl")
if p.exists():
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    print("rows:", len(rows))
    for mode in ["direct", "sandbox"]:
        sub = [r for r in rows if r["mode"] == mode]
        if sub:
            print(f"  {mode:8s} n={len(sub):3d} "
                  f"acc={sum(r['correct'] for r in sub)/len(sub):.3f} "
                  f"tok={sum(r['generated_tokens'] for r in sub)/len(sub):7.0f}")
    print(collections.Counter(r["stop_reason"] for r in rows).most_common())

## Status at end of session

The sweep was **still running** when the session ended: 38/96 `direct` episodes
complete, the `sandbox` arm not yet started. Those partial numbers are not a
result and are not reported as one — the paired comparison needs both arms over
the same items.

`results.jsonl` is resumable, so re-running the same command continues from
where it stopped rather than starting over.